# 5A. Optimización de modelos básicos

## 1. Introducción
En el cuaderno anterior extrajimos conclusiones vitales para la arquitectura de nuestro proyecto, el tamaño de la ventana, cómo dividir los datos según si queremos predecir partidos de la Premier o de LaLiga y el uso o no de técnicas de selección de varibales.

Sin embargo, realizamos las pruebas sobre modelos de regresión logística y ramdom forest con hiperparámetros cualquiera, sin pararnos a comprobar cuáles son mejores. En este cuaderno dejaremos atrás la exploración general y nos centraremos en optimizar los hiperparámetros de nuestros modelos base. Para ello, utilizaremos la técnica de **búsqueda en rejilla**, que probará todas las combinaciones posibles de los valores dados para los hiperparámetros de los modelos y devolverá la mejor. Por supuesto, haremos uso de fast-forward validation para no utilizar datos del futuro para las predicciones.


In [6]:
import pandas as pd
import numpy as np
import sys
import os
import joblib 

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer


sys.path.append(os.path.abspath('../src'))
from model_utils import walk_forward_split, calculate_metrics

import warnings
warnings.filterwarnings('ignore')

## 2. Carga de Datos
Excluimos las mismas columnas que en el cuaderno 4, es decir, identificadores, nombres de equipos, goles en el partido...

In [7]:
df = pd.read_csv('../data/processed/train_features.csv')

target_col = 'resultado_cod'
col_temporada = 'anio_fin_temporada'

cols_no_predictivas = [
    'id', 'fecha', 'local', 'visitante', 'division', col_temporada,
    'resultado', 'resultado_cod', 'goles_local', 'goles_visitante', 'puntos_local', 'puntos_visitante'
]

## 3. Adaptadores para GridSearchCV
El algoritmo GridSearchCV es una herramienta muy potente para la búsqueda exhaustiva de hiperparámetros, pero no es del todo compatible con la arquitectura que establecimos en el cuaderno anterior. 

Dado que nuestra arquitectura final es asimétrica (entrenamos con dos ligas conjuntas, pero queremos medir el rendimiento solo en LaLiga) y nuestra métrica objetivo es probabilística y multiclase (AUC-PR Macro), necesitamos implementar dos "adaptadores" que permitan a GridSearchCV funcionar correctamente bajo estas circunstancias:

### 1. El problema del entrenamiento con ambas ligas
Si introducimos los datos de entrenamiento completos directamente en el optimizador, este entrenará con las temporadas pasadas de Inglaterra y España, pero evaluará la nota sobre el futuro de ambas competiciones combinadas, mientras que queremos medir el rendimiento exclusivamente sobre el futbol español.

Para solucionarlo, hemos construido un generador (basado en nuestra implementación de fast-forward validation original) que actúa como un filtro asimétrico, es decir, en la fese de entrenamiento devuelve los índices de los datos de entrenamiento intactos, para que el modelo aprenda de todo el volumen estadístico disponible (LaLiga + Premier).
Posteriormente, en la fase de evaluación, intercepta los índices del bloque de prueba y elimina cualquier fila que pertenezca a la Premier League, garantizando así que la puntuación de cada hiperparámetro refleje única y exclusivamente su capacidad de generalización sobre la competición que nos interesa.

### 2. El problema de la métrica AUC-PR
Para que GridSearchCV elija los mejores hiperparámetros, necesita una función de puntuación. Para la competición requerimos maximizar el **AUC-PR Macro**, sin embargo, Scikit-Learn no dispone de un evaluador nativo directo para este caso como ya hemos visto anteriomente, por lo que tenemos que inyectarle al algoritmo nuestra porpia implementación.
Sin embargo, cuando le pedimos al optimizador que evalúe por probabilidades, este solo es capaz de entregar dos parámetros a nuestra función, los datos reales (y_true) y los porcentajes (y_proba). Al no entregar las predicciones estáticas (y_pred), nuestra función `calculate_metrics` (que calcula AUC-PR Macro y F1-score) lanzaría un error.

Para solucionarlo, hemos diseñado una función que inyecta los datos faltantes. Recibe y_true e y_proba del Grid, y llama a nuestra función interna "calculate_metrics" y rellena el hueco vacío pasando las etiquetas reales como si fueran predicciones (y_pred = y_true). Esto evidentemente no tiene sentido matemático ni lógico ya que se generarí un cálculo de la métrica F1-score perfecto. Sin embargo, como la métrica que vamos a utilizar para medir el rendimiento es AUC-PR Macro y para el cálculo de la misma no se necesita este parámetro, podremos extraer y devolver de forma aislada el AUC-PR Macro calculado de manera correcta. 

Con la función "make_scorer", convertimos este adaptador en un objeto oficial que obliga al motor de búsqueda en rejilla a optimizar los parámetros de la Regresión Logística basándose estrictamente en las reglas matemáticas de la competición.

Cabe destacar que fue mediante herramientas de IA generativa que se llegó a la conclusión de la necesidad de implementar estos adaptadores, además de apoyarnos en ella para su implementación.

In [16]:
def walk_forward_split_cruzado(df, col_temporada, window_type, window_size):
    """
    Generador que retorna Train (Todo) y Test (SOLO LaLiga).
    """
    base_splitter = walk_forward_split(df, col_temporada=col_temporada, window_type=window_type, window_size=window_size)
    
    for train_idx, test_idx in base_splitter:
        test_idx_laliga = df.loc[test_idx][df.loc[test_idx]['division'] == 'ESP_LaLiga'].index
        yield train_idx, test_idx_laliga

def auc_pr_scorer_wrapper(y_true, y_proba, **kwargs):
    """
    Envuelve nuestra función nativa de utilidades. Como GridSearchCV con needs_proba=True 
    solo pasa y_proba, inyectamos y_true como y_pred temporal (solo nos interesa el AUC-PR que devuelve).
    """
    _, auc_pr = calculate_metrics(y_true, y_pred=y_true, y_proba=y_proba)
    return auc_pr

auc_pr_scorer = make_scorer(auc_pr_scorer_wrapper, response_method='predict_proba')


## 4. Optimización para la Premier League

Procedemos ahora a realizar la optimización de hiperparámetros de los algoritmos de aprendizaje para la Premier League, por supuesto teniendo en cuenta todas las conclusiones a las que hemos llegado hasta ahora.

Primero, definimos un Pipeline de Scikit-Learn para: 
* Escalar los datos (para que tengan media 0 y varianza 1).
* Establecer el selector de variables en "passthrough", lo que provoca que los datos "pasen de largo". Lo dejamos así para comparar entre no usar selector de variables o usar PCA. Aunque comprobamos en el cuaderno 4 que el uso de PCA debería dar mejor resultado, vamos a volver a tener en cuenta la opción de no utilizar PCA, ya que como vimos en el análisis del impacto de las variables, todas aportan valor y no hay ninguna que sobre, por lo que podría darse alguna sorpresa al variar los hiperparámetros de los modelos.
* Elegir el modelo (LogisticRegression para regresión logística y RandomForestClassifier para random forest).

### Regresión Logística
Para nuestro modelo LogisticRegression, iteramos sobre el parámetro **C** (representa el **inverso de la fuerza de regularización**, que al tener un valor bajo, aplica una penalización grande a los pesos que el modelo establece para cada variable. Es decor, lo obliga a buscar reglas generales y robustas, ignorando las anomalías. Es bueno para prevenir el sobreajuste. Con valores altos ocurre lo contraio, se da libertad al modelo para ajustarse al máximo a los datos de entrenamiento, lo que aumenta el riesgo de memorizar el ruido estadístico.)

Nuestro objetivo es por tanto encontrar  el "punto dulce", es decir, un equilibrio donde el algoritmo capte los patrones reales de la liga, sin llegar a memorizar los resultados atípicos.

### Random Forest
Para el modelo RandomForestClassifier, la rejilla explora dos hiperparámetros fundamentales para controlar la complejidad de los árboles:
* **n_estimator**: Determina el número de árboles que formarán el "bosque". Un mayor número incrementa la estabilidad y suaviza las predicciones de probabilidad, aunque a costa de un mayor tiempo de computación.
* **max_depth**: Limita la profundidad máxima de cada árbol. Al restringir el crecimiento a niveles bajos, evitamos que el algoritmo memorice situaciones atípicas del conjunto de entrenamiento, forzándolo a aprender reglas de decisión amplias y generalizables.

### Exportación del mejor modelo

Guardamos el modelo ganador con la biblioteca joblib, que exporta el modelo ganador de la memoria RAM a un archivo físico en el disco duro con extensión .pkl. Esto permitirá cargarlo en otros cuadernos para realizar predicciones.


In [14]:
df_premier = df[df['division'] == 'ENG_Premier'].copy().reset_index(drop=True)
X_prem = df_premier.drop(columns=cols_no_predictivas)
y_prem = df_premier[target_col]

cv_prem = list(walk_forward_split(df_premier, col_temporada=col_temporada, window_type='expanding', window_size=1))

# Regresión logística
pipeline_lr_prem = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', 'passthrough'),
    ('model', LogisticRegression(class_weight='balanced', random_state=42))
])

param_grid_lr = {
    'selector': ['passthrough', PCA(n_components=0.95, random_state=42)],
    'model__C': [0.01, 0.1, 1.0, 10.0],
}

grid_lr_prem = GridSearchCV(pipeline_lr_prem, param_grid_lr, cv=cv_prem, scoring=auc_pr_scorer, n_jobs=-1)
grid_lr_prem.fit(X_prem, y_prem)

print(f"[Regresion Logistica] Mejor AUC-PR: {grid_lr_prem.best_score_:.4f}")
print(f"[Regresion Logistica] Mejores Parámetros: {grid_lr_prem.best_params_}\n")

# Random Forest
pipeline_rf_prem = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', 'passthrough'),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_grid_rf = {
    'selector': ['passthrough', PCA(n_components=0.95, random_state=42)],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 7],
}

grid_rf_prem = GridSearchCV(pipeline_rf_prem, param_grid_rf, cv=cv_prem, scoring=auc_pr_scorer, n_jobs=-1)
grid_rf_prem.fit(X_prem, y_prem)

print(f"[Random Forest] Mejor AUC-PR : {grid_rf_prem.best_score_:.4f}")
print(f"[Random Forest] Mejores Parámetros: {grid_rf_prem.best_params_}\n")

# Exportamos
if grid_lr_prem.best_score_ > grid_rf_prem.best_score_:
    mejor_modelo_premier = grid_lr_prem.best_estimator_
    nombre_archivo = '../models/regresion_logistica_premier.pkl'
else:
    mejor_modelo_premier = grid_rf_prem.best_estimator_
    nombre_archivo = '../models/random_forest_premier.pkl'

joblib.dump(mejor_modelo_premier, nombre_archivo)


[Regresion Logistica] Mejor AUC-PR: 0.4853
[Regresion Logistica] Mejores Parámetros: {'model__C': 0.01, 'selector': 'passthrough'}

[Random Forest] Mejor AUC-PR : 0.4795
[Random Forest] Mejores Parámetros: {'model__max_depth': 5, 'model__n_estimators': 200, 'selector': 'passthrough'}



['../models/regresion_logistica_premier.pkl']

## 5. Optimización para LaLiga

Realizamos la optimización de hiperparámetros en este caso para LaLiga. Lo realizamos de la misma manera que en el apartado anterior (incluso reutilizaremos el pipeline inicial para escalar los datos y elegir el modelo). La principal diferencia en este caso es que, como comprobamos en el cuaderno anterior que el uso de técnicas de selección de características tenía un impacto negativo, mantendremos el selector en "passthrough" para que no se aplique ninguna.

Además, utilizaremos los "adaptadores" explicados en el punto 3 para poder utilizar correctamente los todos los datos para entrenar y sólo los de LaLiga para evaluar.

In [17]:
df_ambos = df.copy().reset_index(drop=True)
X_ambos = df_ambos.drop(columns=cols_no_predictivas)
y_ambos = df_ambos[target_col]

cv_laliga = list(walk_forward_split_cruzado(df_ambos, col_temporada=col_temporada, window_type='expanding', window_size=1))

# Regresión Logística
param_grid_lr_laliga = {
    'selector': ['passthrough'],
    'model__C': [0.01, 0.1, 1.0, 10.0],
}



grid_lr_laliga = GridSearchCV(pipeline_lr_prem, param_grid_lr_laliga, cv=cv_laliga, scoring=auc_pr_scorer, n_jobs=-1)
grid_lr_laliga.fit(X_ambos, y_ambos)

print(f"[LogReg LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): {grid_lr_laliga.best_score_:.4f}")
print(f"[LogReg LaLiga] Mejores Parámetros: {grid_lr_laliga.best_params_}\n")


# Random Forest

param_grid_rf_laliga = {
    'selector': ['passthrough'],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 7],
}
grid_rf_laliga = GridSearchCV(pipeline_rf_prem, param_grid_rf_laliga, cv=cv_laliga, scoring=auc_pr_scorer, n_jobs=-1)
grid_rf_laliga.fit(X_ambos, y_ambos)

print(f"[RF LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): {grid_rf_laliga.best_score_:.4f}")
print(f"[RF LaLiga] Mejores Parámetros: {grid_rf_laliga.best_params_}\n")

# Exportamos
if grid_lr_laliga.best_score_ > grid_rf_laliga.best_score_:
    mejor_modelo_laliga = grid_lr_laliga.best_estimator_
    nombre_archivo = '../models/regresión_logística_laliga.pkl'
else:
    mejor_modelo_laliga = grid_rf_laliga.best_estimator_
    nombre_archivo = '../models/random_forest_laliga.pkl'

joblib.dump(mejor_modelo_laliga, nombre_archivo)


[LogReg LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): 0.4723
[LogReg LaLiga] Mejores Parámetros: {'model__C': 0.01, 'selector': 'passthrough'}

[RF LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): 0.4689
[RF LaLiga] Mejores Parámetros: {'model__max_depth': 5, 'model__n_estimators': 200, 'selector': 'passthrough'}



['../models/regresión_logística_laliga.pkl']

## 6. Conclusión
Finalmente hemos obtenido dos archivos .pkl en la carpeta models/, uno con el mejor modelo para LaLiga y el otro con el mejor modelo para la Premier League. A continuación entrenaremos modelos más complejos en el cuaderno 5B y los compararemos con estos en el cuaderno 6, donde veremos cuáles predicen mejor los datos de prueba.


Como ya se intuía en el cuaderno 4, la **Regresión Logística** se corona como el algoritmo indiscutible para este problema. Supera al Random Forest tanto en LaLiga como en la Premier, alcanzando  **0.4853** en la Premier League, lo cuál supone una mejora de 0,0024 respecto a lo que probamos en el cuaderno 4 y un **0.4723** en el modelo global evaluado sobre LaLiga, lo que supone una meejora de 0,0014 respecto a lo obtenido en el cuaderno 4. Entendemos por tanto que la naturaleza lineal de la regresión logística es la herramienta más adecuada para estimar probabilidades bien calibradas en un entorno de alta incertidumbre.

En cuanto a los hiperparámetros, se ha seleccionado la misma configuración en ambos casos(`{'model__C': 0.01, 'selector': 'passthrough'}`)
* En cuanto al parámetro C, el algoritmo exige el máximo "castigo", lo que confirma que el fútbol tiene un gran nivel de ruido estadístico. Al usar un C tan bajo, forzamos al modelo a crear reglas matemáticas muy generales, impidiendo que memorice resultados atípicos.
* En cuánto a la selección de variables, **se descarta definitivamente** el uso de PCA en ambos modelos, lo cual es muy interesante. Después del análisis en el cuaderno 4, nos esperábamos que el modelo de LaLiga funcionara mejor sin PCA y así ha sido. Sin embargo, esperábamos que el modelo de la Premier funcionara mejor con él. Interpretamos que con PCA se perdían pequeños matices que, aunque no tuvieran mucho impacto, han terminado marcando levemente la diferencia al bajar el valor de C. Lo que si se confirma es la conclusión obtenida en el análisis del impacto de las variables en el cuaderno 4, es decir, que **todas las variables de nuestros datos aportan valor predictivo**.

Por último analizaremos, aunque de manera más breve ya que ha quedado en segundo lugar, el resultado de la optimización del random forest. Es revelador analizar los parámetros ganadores (`{'max_depth': 5, 'n_estimators': 200}`), ya que comprobamos que al elegir una profundidad máxima relativamente baja, el árbol corrobora la necesidad de evitar ramas complejas que memoricen el ruido. Además, exige el máximo número de árboles probado para generar predicciones probabilísticas a través de un gran consenso, suavizando así la varianza natural del deporte.

Es decir, esto concuerda con lo que podemos interpretar de los resultados de la regresión logística: **los modelos funcionan mejor con hiperparámetros que generalizan más sobre los datos de entrenamiento**, evitando sobreajustes o memorización de situaciones puntuales, lo que nos indica la existencia de patrones futbolísticos generales que se han debido repetir durante los 24 años de Liga y Premier que analizamos.

